# Project 8 — Local Blog-to-Thread Converter
## Repurpose Long-Form Content into Threads, Posts, and Emails

**Stack:** Ollama · LangChain · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.5)

blog_post = """
# Why RAG Beats Fine-Tuning for Most Enterprise Use Cases

The debate between Retrieval-Augmented Generation and fine-tuning has evolved
significantly. While fine-tuning creates specialized models, RAG offers advantages
that make it the better default choice for most enterprise applications.

**Cost Efficiency**: Fine-tuning requires curating datasets, running training jobs,
and maintaining multiple model versions. RAG needs only a vector database and an
embedding pipeline — often 10x cheaper to set up and maintain.

**Data Freshness**: Fine-tuned models are frozen at training time. When your company
policies, products, or documentation change, you need to retrain. RAG systems
update instantly — just re-embed the new documents.

**Transparency**: RAG provides source citations. Users can verify answers against
the original documents. Fine-tuned models are black boxes — you can't trace where
an answer came from.

**Composability**: A single RAG system can serve multiple domains by swapping
document collections. Fine-tuning requires separate models for each domain.

**When Fine-Tuning Wins**: Style/tone adaptation, consistent formatting requirements,
and high-throughput latency-sensitive applications where retrieval adds overhead.

The pragmatic approach: Start with RAG, measure gaps, and fine-tune only when
you have clear evidence that RAG alone can't meet your quality bar.
"""
print(f"Blog post loaded: {len(blog_post)} chars")

## Step 2 — Convert to Twitter/X Thread

In [ ]:
thread_prompt = ChatPromptTemplate.from_messages([
    ("system", """Convert blog posts into engaging Twitter/X threads. Rules:
- First tweet: hook that makes people want to read more
- 5-10 tweets total, each under 280 characters
- Use numbered format (1/, 2/, etc.)
- Include a mix of insights, data points, and opinions
- End with a takeaway or call to action
- Use line breaks for readability"""),
    ("human", "Convert this blog post to a Twitter thread:\n\n{blog}")
])

thread_chain = thread_prompt | llm | StrOutputParser()
thread = thread_chain.invoke({"blog": blog_post})
print("TWITTER THREAD:")
print(thread)

## Step 3 — Convert to LinkedIn Post

In [ ]:
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", """Convert blog posts into LinkedIn posts. Rules:
- Opening hook (first line visible before 'See more')
- Professional but conversational tone
- Use bullet points or numbered lists
- Include a personal insight or opinion
- End with a question to drive engagement
- 150-300 words
- Add relevant hashtags"""),
    ("human", "Convert this to a LinkedIn post:\n\n{blog}")
])

linkedin_chain = linkedin_prompt | llm | StrOutputParser()
post = linkedin_chain.invoke({"blog": blog_post})
print("LINKEDIN POST:")
print(post)

## Step 4 — Convert to Newsletter Email

In [ ]:
email_prompt = ChatPromptTemplate.from_messages([
    ("system", """Convert blog posts into newsletter emails. Rules:
- Compelling subject line
- Personal greeting
- TL;DR at the top (3 bullets)
- Expanded discussion with examples
- Clear CTA (reply, share, or link)
- Casual but authoritative tone
- Under 500 words"""),
    ("human", "Convert this blog to a newsletter email:\n\n{blog}")
])

email_chain = email_prompt | llm | StrOutputParser()
newsletter = email_chain.invoke({"blog": blog_post})
print("NEWSLETTER EMAIL:")
print(newsletter)

## Step 5 — All-Formats Pipeline

In [ ]:
formats = {
    "twitter_thread": thread_chain,
    "linkedin_post": linkedin_chain,
    "newsletter_email": email_chain,
}

print("=== CONTENT REPURPOSING PIPELINE ===")
for fmt_name, chain in formats.items():
    print(f"\n{'='*60}")
    print(f"FORMAT: {fmt_name}")
    print('='*60)
    result = chain.invoke({"blog": blog_post})
    print(result[:300] + "..." if len(result) > 300 else result)
    print(f"\n[{len(result)} chars generated]")

## What You Learned
- **Platform-specific content adaptation** (Twitter, LinkedIn, Email)
- **Tone and format constraints** via system prompts
- **Batch content repurposing** pipeline